# 01 - Data Ingestion

**Objective:** Load the Cleveland Heart Disease dataset, handle missing values, encode the target, and save processed output.

**Dataset:** Cleveland Heart Disease (UCI)

**Steps:**
1. Load raw data from `data/raw/processed.cleveland.data`
2. Handle missing values and convert types
3. Binarize target variable (num 0→0, 1-4→1)
4. Check data quality
5. Save processed data


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


### Dataset Attributes

Understanding each column helps you make better decisions during cleaning and feature engineering.

- **age:** Age in years
- **sex:** 1 = Male, 0 = Female
- **cp:** Chest pain type (1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic)
- **trestbps:** Resting blood pressure (mm Hg)
- **chol:** Serum cholesterol (mg/dl)
- **fbs:** Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- **restecg:** Resting ECG results (0 = normal, 1 = ST-T wave abnormality, 2 = left ventricular hypertrophy)
- **thalach:** Maximum heart rate achieved
- **exang:** Exercise-induced angina (1 = yes, 0 = no)
- **oldpeak:** ST depression induced by exercise relative to rest
- **slope:** Slope of peak exercise ST segment (1 = upsloping, 2 = flat, 3 = downsloping)
- **ca:** Number of major vessels colored by fluoroscopy (0–3)
- **thal:** Thalassemia (3 = normal, 6 = fixed defect, 7 = reversible defect)
- **num:** Diagnosis of heart disease (0 = no disease, 1–4 = increasing severity) — what we binarize to predict


In [ ]:
# TODO: Load the raw CSV data into a DataFrame
# The Cleveland dataset has no header row, so pass column names manually.
# Use pd.read_csv() with header=None and names=col_names.

RAW_DIR = Path("../data/raw")

col_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]

# df = pd.read_csv(RAW_DIR / "processed.cleveland.data", header=None, names=col_names)

# TODO: After loading, inspect the data to make sure it looks right
# Use .shape to check the row and column count,
# and .head() to preview the first few rows.
# Then check data types with .info() — note that ca and thal may be strings
# because missing values are encoded as "?" in the raw file.

# print(f"Shape: {df.shape}")
# print(df.head())
# df.info()


In [ ]:
# TODO: Replace "?" with NaN and drop rows with missing values
# Some values in ca and thal are encoded as "?" to indicate missing data.
# Replace these with pd.NA, then drop any rows that still have NaN.
# Always reset the index after dropping rows so the DataFrame stays sequential.

# TODO: Convert ca and thal to numeric types
# After dropping NaN rows, ca and thal are still strings. Cast them to float
# so they can be used as numeric features in the model.

# df = df.replace("?", pd.NA)
# df = df.dropna().reset_index(drop=True)
# df["ca"] = df["ca"].astype(float)
# df["thal"] = df["thal"].astype(float)

# TODO: Verify the final shape after cleaning
# print(f"Clean shape: {df.shape}")


In [ ]:
# TODO: Binarize the target variable
# The raw "num" column is ordinal (0–4), where 0 means no heart disease
# and 1–4 represent increasing severity. For binary classification we
# simplify this: 0 → 0 (no disease), 1–4 → 1 (disease present).
# Create a new "target" column from this mapping, then drop the original "num".

# df["target"] = (df["num"] > 0).astype(int)
# df = df.drop(columns=["num"])

# TODO: Check the class distribution
# Use .value_counts() on the target column to see the balance.
# This tells you what proportion of patients have heart disease.

# print(df["target"].value_counts())
# print(df["target"].value_counts(normalize=True))


In [ ]:
# TODO: Verify there are no remaining missing values
# After cleaning, confirm the dataset is complete with df.isnull().sum().sum().

# TODO: Check the final data types
# All feature columns should be float64, and the target column should be int64.
# Use df.dtypes to verify.

# print(f"Missing values: {df.isnull().sum().sum()}")
# print(df.dtypes)


In [ ]:
# TODO: Save the processed data to data/processed/clean_data.csv
# Use df.to_csv() with index=False so we don't write the DataFrame index as a column.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df.to_csv(PROCESSED_DIR / "clean_data.csv", index=False)
# print("Clean data saved successfully")


In [ ]:
# TODO: Verify the saved file by loading it back into a new DataFrame
# This step confirms the file wasn't corrupted during the write process.
# Compare the shape of the loaded data to the original shape.

# df_check = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# print(f"Loaded data shape: {df_check.shape}")
# assert df_check.shape == df.shape, "Shape mismatch after save-load cycle"
# print("Verification passed: data round-trips correctly")


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table 
and several **dimension** tables, linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the Cleveland heart disease data,
split into three CSV files that simulate a hospital database:

| File | Content | Key column |
|------|---------|------------|
| `patient.csv` | Demographics (age, sex) | `sample_id` |
| `clinical.csv` | Lab and test results | `clinical_id` |
| `outcome.csv` | Final diagnosis | `outcome_id` |

Some IDs appear in only some tables — exactly like a real database where 
test results may be missing or extra records exist for follow-up patients. 
Your job is to re-assemble the full dataset.

In [ ]:
# Load the three relational tables
RELATIONAL_DIR = Path("../data/raw/relational")

patient = pd.read_csv(RELATIONAL_DIR / "patient.csv")
clinical = pd.read_csv(RELATIONAL_DIR / "clinical.csv")
outcome = pd.read_csv(RELATIONAL_DIR / "outcome.csv")

# TODO: Inspect each table
# Check the shape and first few rows of each DataFrame.
# How many IDs does each table have? Are the ID ranges the same?

# print(f"Patient: {patient.shape}")
# print(f"Clinical: {clinical.shape}")
# print(f"Outcome: {outcome.shape}")
# print(patient.head())
# print(clinical.columns.tolist())


In [ ]:
# TODO: Merge patient with clinical results
# Use pd.merge() with left_on="sample_id" and right_on="clinical_id".
# Start with a LEFT JOIN so you keep all patients.
# Count how many rows have NaN — those are patients without lab results.

# merged = pd.merge(patient, clinical, left_on="sample_id", right_on="clinical_id", how="left")
# print(f"After merge: {merged.shape}")
# print(f"Missing clinical: {merged.isna().any(axis=1).sum()}")

# TODO: Try an INNER JOIN. How many rows do you lose? Why?


In [ ]:
# TODO: Merge in the outcome table
# Chain a second merge to bring in the diagnosis target.

# full = pd.merge(merged, outcome, left_on="sample_id", right_on="outcome_id", how="left")
# print(full.head())

# TODO: Check how many rows are missing the target diagnosis
# missing_outcome = full['target'].isna().sum()
# print(f"Missing outcome: {missing_outcome}")


In [ ]:
# TODO: Compare to the original single-table shape
# The original cleveland.data has 297 clean rows with all 13 features + target.
# After splitting and rejoining, how many rows have everything?

# complete = full.dropna()
# print(f"Complete rows after join: {len(complete)}")

# HINT: The difference tells you how many records are missing from each table.
# Which table had the most missing records?


### Exercises

1. **Explore encoding options**: The target is binarized from ordinal num (0→0, 1–4→1). What would change if you treated num as a multi-class target instead? How many classes would you have?
2. **Experiment with missing value handling**: Intentionally introduce additional NaN values with `df.loc[:10, "chol"] = np.nan`, then try both `dropna()` and `fillna(df["chol"].mean())` and compare the resulting shapes.
3. **Save in Parquet format**: Use `df.to_parquet()` instead of CSV and compare file sizes. Parquet is often faster and more space-efficient.
4. **Feature engineering idea**: Create a new feature `max_hr_ratio = thalach / age`. Does this ratio capture cardiovascular fitness differently from either raw measurement?
5. **What would break?**: If the raw data file used "NA" instead of "?" for missing values, which part of this notebook would fail? What if a new column was added?
6. **Which JOIN?**: You used LEFT JOINs to assemble the relational tables. Try replacing them with INNER JOINs. How many rows do you lose overall? Why?
7. **Fake data detective**: The clinical and outcome tables contain artificially generated rows (IDs > 297). Can you identify them? What heuristic did you use?
8. **Save the reloaded version**: After merging all three relational tables back, save the result to `data/processed/clean_data.csv`.